# Notebook d'exploration — Assistant RAG
Ce fichier s'ouvre comme un notebook Jupyter directement dans VS Code
(chaque bloc "# %%" est une cellule exécutable indépendamment, avec
CTRL+ENTRÉE ou en cliquant sur "Run Cell" au-dessus de la cellule).

Objectif : comprendre le pipeline RAG étape par étape, AVANT de le
retrouver organisé en fonctions réutilisables dans app/rag.py.

In [ ]:
# --- 1. Imports ---
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

In [ ]:
# --- 2. Charger le PDF de l'entreprise ---
# Placez votre document dans le dossier data/ AVANT d'exécuter cette cellule,
# puis remplacez le nom de fichier ci-dessous.
PDF_PATH = "data/mon_document.pdf"  # <-- à adapter

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()  # une "Document" LangChain par page du PDF

print(f"Nombre de pages chargées : {len(pages)}")
print("--- Aperçu de la première page ---")
print(pages[0].page_content[:500])

In [ ]:
# --- 3. Découper le texte en morceaux ("chunking") ---
# Pourquoi découper ? Un LLM ne peut pas "lire" tout un document d'un coup
# de façon fiable, et on veut pouvoir retrouver UNIQUEMENT le passage
# pertinent pour une question donnée. On coupe donc le texte en petits
# blocs ("chunks") que l'on pourra chercher individuellement.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # ~1000 caractères par morceau
    chunk_overlap=150,   # chevauchement pour ne pas couper une idée en deux
)
chunks = splitter.split_documents(pages)

print(f"Nombre de chunks créés : {len(chunks)}")
print("--- Exemple de chunk ---")
print(chunks[0].page_content)
print(chunks[0].metadata)  # contient la page d'origine -> utile pour citer les sources

In [ ]:
# --- 4. Créer les embeddings (vecteurs numériques) ---
# Un embedding transforme un texte en une liste de nombres qui capture son
# "sens". Deux textes proches en signification auront des vecteurs proches.
# On utilise "nomic-embed-text", servi localement par Ollama :
# gratuit, tourne sur CPU, léger (~270 Mo en RAM).
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Test rapide : embedder une phrase
test_vector = embeddings.embed_query("Quelle est la politique de congés ?")
print(f"Longueur du vecteur généré : {len(test_vector)}")

In [ ]:
# --- 5. Stocker les vecteurs dans une base vectorielle (Chroma) ---
# Chroma est une base de données spécialisée dans la recherche par
# similarité de vecteurs. Elle tourne en local, sans serveur externe,
# et sauvegarde les données sur disque via persist_directory.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db_notebook",  # dossier séparé de la prod, pour expérimenter
)
print("Base vectorielle créée et sauvegardée sur disque.")

In [ ]:
# --- 6. Tester la recherche (retrieval) ---
# On cherche les chunks les plus proches sémantiquement de la question.
question = "Quelle est la politique de congés de l'entreprise ?"
resultats = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(resultats, 1):
    print(f"\n--- Résultat {i} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])

In [ ]:
# --- 7. Générer une réponse avec le LLM local ---
# ChatOllama envoie un prompt au modèle qui tourne localement via Ollama.
# "llama3.2:3b" est un bon compromis qualité / vitesse pour 16 Go de RAM
# (alternative possible : "qwen2.5:3b-instruct" ou "phi3.5:3.8b").
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

contexte = "\n\n".join(doc.page_content for doc in resultats)
prompt = f"""Réponds à la question en te basant uniquement sur ce contexte :

{contexte}

Question : {question}
Réponse :"""

reponse = llm.invoke(prompt)
print(reponse.content)

## Bilan
Vous venez de reconstituer à la main le pipeline complet :

**PDF → chunks → embeddings → recherche (retrieval) → prompt → réponse du LLM**

C'est exactement ce que fait `app/rag.py`, mais organisé en fonctions
réutilisables pour être appelé par l'API FastAPI (`app/main.py`).